# 3-stage: слайд-шоу предсказаний модели

Положи модель в `3-stage/best_model.pth`. КТ-срезы и маски остаются в `preprocessed_npy/` или `prepared_npy/`.

Notebook загружает все срезы выбранного пациента, выполняет inference и формирует статичные примеры, графики и HTML5-видео. На кадрах отдельно показываются метрики текущего среза и агрегированные метрики всего выбранного объёма пациента. Если на срезе нет опухоли и модель тоже ничего не выделила, per-slice метрики показываются как `n/a`, чтобы пустые кадры не выглядели как ошибка модели.


## 1. Импорты и настройки

Поменяй `DATASET_DIR` и `SPLIT`, если нужен другой набор данных. Пациент будет случайно выбран из `manifest.csv`.


In [ ]:
from pathlib import Path
import csv
import json
import importlib.util
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import torch
from IPython.display import HTML
from celluloid import Camera


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "2-stage" / "model.py").exists() and (candidate / "3-stage").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден 2-stage/model.py. Запусти notebook из репозитория Lung-Tumor-Segmentation."
    )


REPO_ROOT = find_repo_root()
CHECKPOINT_PATH = REPO_ROOT / "3-stage" / "best_model.pth"

# Выбор серии срезов для слайд-шоу.
DATASET_DIR = REPO_ROOT / "preprocessed_npy"  # или REPO_ROOT / "prepared_npy"
SPLIT = "test"
POSITIVE_CASE_ONLY = True
RANDOM_SEED = None  # укажи число, например 2004, для повторяемого выбора пациента
INFERENCE_BATCH_SIZE = 8

MANIFEST_PATH = DATASET_DIR / "manifest.csv"
for path, label in [
    (CHECKPOINT_PATH, "model"),
    (MANIFEST_PATH, "manifest"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден {label}: {path}")

with MANIFEST_PATH.open(newline="", encoding="utf-8-sig") as manifest_file:
    manifest_rows = list(csv.DictReader(manifest_file))

required_columns = {"split", "case_id", "has_tumor"}
missing_columns = required_columns - set(manifest_rows[0] if manifest_rows else [])
if missing_columns:
    raise ValueError(f"В manifest.csv отсутствуют колонки: {sorted(missing_columns)}")

split_rows = [row for row in manifest_rows if row["split"].strip() == SPLIT]
if not split_rows:
    raise ValueError(f"В manifest.csv нет строк для split={SPLIT!r}")

positive_values = {"1", "true", "yes"}
if POSITIVE_CASE_ONLY:
    candidate_case_ids = sorted({
        row["case_id"].strip()
        for row in split_rows
        if row["has_tumor"].strip().lower() in positive_values
    })
else:
    candidate_case_ids = sorted({row["case_id"].strip() for row in split_rows})

if not candidate_case_ids:
    raise ValueError(f"Для split={SPLIT!r} не найдены подходящие пациенты")

CASE_ID = random.Random(RANDOM_SEED).choice(candidate_case_ids)
IMAGE_DIR = DATASET_DIR / SPLIT / CASE_ID / "images"
MASK_DIR = DATASET_DIR / SPLIT / CASE_ID / "masks"

for path, label in [
    (IMAGE_DIR, "images directory"),
    (MASK_DIR, "masks directory"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден {label}: {path}")

MODEL_FILE = REPO_ROOT / "2-stage" / "model.py"
spec = importlib.util.spec_from_file_location("lung_unet_model", MODEL_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f"Не удалось загрузить model.py: {MODEL_FILE}")
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
build_model = model_module.build_model

image_paths = sorted(IMAGE_DIR.glob("*.npy"))
if not image_paths:
    raise FileNotFoundError(f"В папке нет .npy-срезов: {IMAGE_DIR}")

mask_paths = [MASK_DIR / path.name for path in image_paths]
missing_masks = [path for path in mask_paths if not path.exists()]
if missing_masks:
    raise FileNotFoundError(f"Не найдены маски для {len(missing_masks)} срезов. Первая: {missing_masks[0]}")

print("model:", CHECKPOINT_PATH)
print("case:", f"{SPLIT}/{CASE_ID}")
print("slices:", len(image_paths))


## 2. Вспомогательные функции


In [ ]:
def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def as_chw_float32(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    if array.ndim == 2:
        array = array[None, :, :]
    elif array.ndim == 3 and array.shape[0] == 1:
        pass
    elif array.ndim == 3 and array.shape[-1] == 1:
        array = np.moveaxis(array, -1, 0)
    else:
        raise ValueError(f"Ожидался массив [H, W] или [1, H, W], получено: {array.shape}")
    return array.astype(np.float32, copy=False)


def masked(mask: np.ndarray):
    return np.ma.masked_where(mask == 0, mask)


def confusion(pred_mask: np.ndarray, gt_mask: np.ndarray):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    return {
        "tp": float(np.logical_and(pred, gt).sum()),
        "fp": float(np.logical_and(pred, ~gt).sum()),
        "fn": float(np.logical_and(~pred, gt).sum()),
    }


def metrics_from_confusion(values, empty_value=np.nan):
    tp, fp, fn = values["tp"], values["fp"], values["fn"]
    eps = 1e-7
    dice_denominator = 2.0 * tp + fp + fn
    pred_pixels = tp + fp
    gt_pixels = tp + fn
    return {
        "dice": empty_value if dice_denominator == 0 else (2.0 * tp) / (dice_denominator + eps),
        "precision": empty_value if pred_pixels == 0 else tp / (pred_pixels + eps),
        "recall": empty_value if gt_pixels == 0 else tp / (gt_pixels + eps),
    }


def format_metric(value) -> str:
    if value is None or not np.isfinite(value):
        return "n/a"
    return f"{value:.3f}"


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


## 3. Загрузка модели и inference всех срезов


In [ ]:
checkpoint = load_checkpoint(CHECKPOINT_PATH)
config = checkpoint.get("config", {})
threshold = float(checkpoint.get("best_threshold", config.get("metrics", {}).get("threshold", 0.5)))
device = choose_device()

model = build_model(config.get("model", {})).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

images = [as_chw_float32(np.load(path, allow_pickle=False)) for path in image_paths]
gt_masks = [(as_chw_float32(np.load(path, allow_pickle=False))[0] > 0.5).astype(np.uint8) for path in mask_paths]

image_shape = images[0].shape
if any(image.shape != image_shape for image in images):
    raise ValueError("Все КТ-срезы пациента должны иметь одинаковый размер")
if any(mask.shape != image_shape[1:] for mask in gt_masks):
    raise ValueError("Размеры КТ-срезов и масок не совпадают")

probabilities = []
with torch.no_grad():
    for start in range(0, len(images), INFERENCE_BATCH_SIZE):
        batch_np = np.stack(images[start:start + INFERENCE_BATCH_SIZE])
        batch = torch.from_numpy(np.ascontiguousarray(batch_np)).to(device)
        logits = model(batch)
        probabilities.extend(torch.sigmoid(logits)[:, 0].detach().cpu().numpy().astype(np.float32))

pred_masks = [(probability >= threshold).astype(np.uint8) for probability in probabilities]
slice_confusions = [confusion(pred, gt) for pred, gt in zip(pred_masks, gt_masks)]
slice_metrics = [metrics_from_confusion(values, empty_value=np.nan) for values in slice_confusions]
volume_confusion = {
    key: sum(values[key] for values in slice_confusions)
    for key in ("tp", "fp", "fn")
}
volume_metrics = metrics_from_confusion(volume_confusion, empty_value=np.nan)

print("device:", device)
print("checkpoint epoch:", checkpoint.get("epoch"))
print("best val dice:", checkpoint.get("best_val_dice"))
print("threshold:", threshold)
print("volume Dice:", volume_metrics["dice"])
print("volume Precision:", volume_metrics["precision"])
print("volume Recall:", volume_metrics["recall"])
print("ground-truth tumor slices:", sum(bool(mask.any()) for mask in gt_masks))
print("predicted tumor slices:", sum(bool(mask.any()) for mask in pred_masks))
print("ground-truth tumor pixels:", sum(int(mask.sum()) for mask in gt_masks))
print("predicted tumor pixels:", sum(int(mask.sum()) for mask in pred_masks))
print("TP / FP / FN:", volume_confusion["tp"], volume_confusion["fp"], volume_confusion["fn"])


## 4. Сравнение моделей на тестовой выборке

График строится по финальным сопоставимым запускам `prepared_npy_patch384`: U-Net, UNet++ и MONAI Attention U-Net.


In [ ]:
RESULT_RUNS = {
    "U-Net": REPO_ROOT / "result" / "unet" / "run_20260526_103729",
    "UNet++": REPO_ROOT / "result" / "unet++" / "run_20260526_110321",
    "Attention U-Net": REPO_ROOT / "result" / "unet_attention" / "run_20260526_115212",
}

metric_names = ["dice", "precision", "recall"]
metric_labels = ["Dice coefficient", "Precision", "Recall"]
model_names = []
test_values = {metric: [] for metric in metric_names}

for model_name, run_dir in RESULT_RUNS.items():
    metrics_path = run_dir / "metrics" / "test_metrics.json"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Не найден test_metrics.json: {metrics_path}")
    metrics = load_json(metrics_path)
    model_names.append(model_name)
    for metric in metric_names:
        test_values[metric].append(float(metrics[metric]))

x = np.arange(len(model_names))
width = 0.24
fig, ax = plt.subplots(figsize=(9, 4.8))
for offset, metric, label in zip([-width, 0, width], metric_names, metric_labels):
    bars = ax.bar(x + offset, test_values[metric], width, label=label)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0, max(max(values) for values in test_values.values()) * 1.25)
ax.set_ylabel("Metric value")
ax.set_title("Comparison of models on the test set")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 5. Динамика обучения моделей

Сплошная линия — train, пунктир — validation. Для validation используется `val_best_dice`, потому что checkpoint выбирался по лучшему Dice после подбора threshold на validation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for model_name, run_dir in RESULT_RUNS.items():
    history_path = run_dir / "metrics" / "history.json"
    if not history_path.exists():
        raise FileNotFoundError(f"Не найден history.json: {history_path}")
    history = load_json(history_path)
    epochs = [row["epoch"] for row in history]

    axes[0].plot(epochs, [row["train_dice"] for row in history], label=f"{model_name} train")
    axes[0].plot(
        epochs,
        [row.get("val_best_dice", row["val_dice"]) for row in history],
        linestyle="--",
        label=f"{model_name} val",
    )

    axes[1].plot(epochs, [row["train_loss"] for row in history], label=f"{model_name} train")
    axes[1].plot(epochs, [row["val_loss"] for row in history], linestyle="--", label=f"{model_name} val")

axes[0].set_title("Dice dynamics")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Dice coefficient")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8, ncol=2)

axes[1].set_title("Loss dynamics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()


## 6. Примеры предсказанных масок сегментации

Показываются информативные тестовые срезы выбранного пациента: исходный КТ-срез, истинная маска и наложение истинной/предсказанной маски.


In [ ]:
N_EXAMPLES = 4


def choose_example_indices(gt_masks, pred_masks, n=4):
    interesting = sorted({
        index
        for index, (gt_mask, pred_mask) in enumerate(zip(gt_masks, pred_masks))
        if gt_mask.any() or pred_mask.any()
    })
    if not interesting:
        return np.linspace(0, len(gt_masks) - 1, min(n, len(gt_masks)), dtype=int).tolist()
    if len(interesting) <= n:
        return interesting
    positions = np.linspace(0, len(interesting) - 1, n)
    return [interesting[int(round(position))] for position in positions]


example_indices = choose_example_indices(gt_masks, pred_masks, N_EXAMPLES)
fig, axes = plt.subplots(len(example_indices), 3, figsize=(11, 3.4 * len(example_indices)))
if len(example_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_index, slice_index in enumerate(example_indices):
    image = images[slice_index][0]
    gt_mask = gt_masks[slice_index]
    pred_mask = pred_masks[slice_index]
    metrics = slice_metrics[slice_index]
    image_name = image_paths[slice_index].stem

    panels = [
        ("CT slice", False, False),
        ("Ground truth", True, False),
        (
            "Prediction overlay\n"
            f"Slice Dice={format_metric(metrics['dice'])}, "
            f"P={format_metric(metrics['precision'])}, "
            f"R={format_metric(metrics['recall'])}",
            True,
            True,
        ),
    ]
    for col_index, (title, show_gt, show_pred) in enumerate(panels):
        ax = axes[row_index, col_index]
        ax.imshow(image, cmap="bone", vmin=0, vmax=1)
        if show_gt:
            ax.imshow(masked(gt_mask), cmap="Blues", alpha=0.55, vmin=0, vmax=1)
        if show_pred:
            ax.imshow(masked(pred_mask), cmap="Reds", alpha=0.55, vmin=0, vmax=1)
        ax.set_title(f"{image_name} | {title}", fontsize=10)
        ax.axis("off")

handles = [
    Patch(facecolor="tab:blue", alpha=0.55, label="Ground truth"),
    Patch(facecolor="tab:red", alpha=0.55, label="Prediction"),
]
fig.legend(handles=handles, loc="lower center", ncol=2)
fig.suptitle(f"Segmentation examples for {SPLIT}/{CASE_ID}", y=0.995)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
plt.show()


## 7. Видео слайд-шоу

Синяя область — истинная маска, красная — предсказание модели. В заголовке кадра отдельно показаны метрики текущего среза и метрики всего выбранного объёма пациента.


In [ ]:
plt.rcParams["animation.embed_limit"] = 100
fig, ax = plt.subplots(figsize=(8, 8))
camera = Camera(fig)
volume_text = (
    f"Volume: Dice={format_metric(volume_metrics['dice'])} | "
    f"Precision={format_metric(volume_metrics['precision'])} | "
    f"Recall={format_metric(volume_metrics['recall'])}"
)

for index, image_path in enumerate(image_paths):
    image = images[index][0]
    gt_mask = gt_masks[index]
    pred_mask = pred_masks[index]
    metrics = slice_metrics[index]
    gt_pixels = int(gt_mask.sum())
    pred_pixels = int(pred_mask.sum())

    ax.imshow(image, cmap="bone", vmin=0, vmax=1)
    ax.imshow(masked(gt_mask), cmap="Blues", alpha=0.55, vmin=0, vmax=1)
    ax.imshow(masked(pred_mask), cmap="Reds", alpha=0.55, vmin=0, vmax=1)
    ax.set_title(
        f"{CASE_ID} | {image_path.stem} | GT pixels={gt_pixels} | Pred pixels={pred_pixels}\n"
        f"Slice: Dice={format_metric(metrics['dice'])} | "
        f"Precision={format_metric(metrics['precision'])} | "
        f"Recall={format_metric(metrics['recall'])}\n"
        f"{volume_text}",
        fontsize=10,
    )
    ax.legend(
        handles=[
            Patch(facecolor="tab:blue", alpha=0.55, label="Ground truth"),
            Patch(facecolor="tab:red", alpha=0.55, label="Prediction"),
        ],
        loc="lower right",
    )
    ax.axis("off")
    camera.snap()

animation = camera.animate(interval=250, repeat=True, blit=True)
plt.close(fig)
HTML(animation.to_html5_video())
